# Inspect synthetic robot recordings with Kalanos

Download a public Kalanos synthetic fixture, run the open-source grader, and inspect its score, decisions and evidence. No robot, GPU, Kalanos API key or private dataset is needed.

**Draft: fresh Colab execution has not yet been verified.** The grader runs inside your Colab VM. It does not call MCP or publish a report.

This is deliberately synthetic test data, not a training-performance benchmark. An action-spikes fixture can have a high aggregate score; inspect findings and coverage rather than treating the score as a training guarantee.

Choose **Runtime → Run all** on a CPU runtime. Installation and downloads require internet. Download results before the temporary runtime expires.


## 1. Set up an isolated grader environment
This avoids replacing packages used by the notebook kernel. Kalanos and Hugging Face Hub are pinned; transitive dependencies are recorded after installation, not fully locked.

In [ ]:
import json
import os
import re
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

from IPython.display import HTML, FileLink, display


WORK = Path(tempfile.mkdtemp(prefix="kalanos-colab-"))
VENV = WORK / "venv"
OUTPUT = WORK / "results"
OUTPUT.mkdir()
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
PYTHON = VENV / "bin" / "python"
KALANOS = VENV / "bin" / "kalanos"
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "kalanos[numeric,video]==0.10.0",
        "huggingface-hub==1.31.0",
    ],
    check=True,
)
lock = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(OUTPUT / "requirements-resolved.txt").write_text(lock)
print("Working directory:", WORK)

## 2. Download the fixture
Default: [synthetic-action-spikes](https://huggingface.co/datasets/kalanosai/synthetic-action-spikes).

A blank revision resolves the current public commit **once** and downloads that immutable revision. Save the printed SHA in `DATASET_REVISION` for identical data on later runs. The release maintainer should pin the validated SHA before publishing this notebook as a reproducible tutorial.

The download is limited to 200 MB. This notebook expects unpacked LeRobot files with `meta/info.json`, not an archive. It stops rather than silently grading README or ground-truth files as recordings.


In [ ]:
DATASET_ID = "kalanosai/synthetic-action-spikes"
DATASET_REVISION = ""  # Set a validated 40-character Hugging Face commit SHA to replay.
MAX_BYTES = 200 * 1024 * 1024

if DATASET_REVISION and not re.fullmatch(r"[a-f0-9]{40}", DATASET_REVISION):
    raise ValueError(
        "Use an exact Hugging Face commit SHA, or leave blank to resolve once."
    )

download_script = r"""import json, pathlib, sys
from huggingface_hub import HfApi, snapshot_download
repo, revision, destination, max_bytes = sys.argv[1:]
api = HfApi(token=False)
info = api.dataset_info(repo, revision=revision or "main", files_metadata=True)
files = info.siblings
if not files or any(f.size is None for f in files):
    raise RuntimeError("Cannot verify repository size; no download performed.")
total = sum(f.size for f in files)
if total > int(max_bytes):
    raise RuntimeError(f"Repository is {total} bytes, above download limit.")
path = snapshot_download(repo_id=repo, repo_type="dataset", revision=info.sha,
                         local_dir=destination, token=False, max_workers=2)
print(json.dumps({"dataset_id":repo, "revision":info.sha,
                  "repository_bytes":total, "local_path":path}))
"""
env = dict(os.environ, HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
raw = subprocess.check_output(
    [
        str(PYTHON),
        "-c",
        download_script,
        DATASET_ID,
        DATASET_REVISION,
        str(WORK / "download"),
        str(MAX_BYTES),
    ],
    text=True,
    env=env,
)
provenance = json.loads(raw.strip().splitlines()[-1])
roots = [
    p.parent.parent for p in Path(provenance["local_path"]).rglob("meta/info.json")
]
if len(roots) != 1:
    raise RuntimeError(
        f"Expected one unpacked LeRobot root, found {len(roots)}. "
        "Inspect repository layout."
    )
DATASET = roots[0]
info = json.loads((DATASET / "meta/info.json").read_text())
if not list((DATASET / "data").rglob("*.parquet")):
    raise RuntimeError(
        "No LeRobot Parquet data found; do not grade metadata as recordings."
    )
provenance.update(
    {
        "declared_format": info.get("codebase_version"),
        "declared_episodes": info.get("total_episodes"),
        "grader_version": "0.10.0",
        "python_version": sys.version,
    }
)
(OUTPUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))

## 3. Grade and inspect the result
The CLI owns score and decision semantics. We display its own summary instead of assuming JSON field names or substituting a historical Hub score.

Exit **0** means the selected gate did not fail; **1** means a valid assessment failed the gate; other codes are treated as execution errors. A gate pass does not mean there are no findings. The default evaluation scope is recorded in the output; confirm it matches your intended use.


In [ ]:
grader_env = dict(os.environ, NO_COLOR="1")
grader_env.pop("KALANOS_API_KEY", None)


def grade_report(suffix):
    report = OUTPUT / f"audit.{suffix}"
    command = [
        str(KALANOS),
        "grade",
        str(DATASET),
        "--report",
        str(report),
        "--fail-on",
        "blocked,review,unknown",
    ]
    result = subprocess.run(command, env=grader_env, capture_output=True, text=True)
    (OUTPUT / f"grader-{suffix}.log").write_text(result.stdout + "\n" + result.stderr)
    print(result.stdout)
    if result.returncode not in (0, 1) or not report.is_file():
        raise RuntimeError(
            f"Grading failed (exit {result.returncode}). " + result.stderr[-3000:]
        )
    print(
        "Gate:", "passed" if result.returncode == 0 else "failed; review the assessment"
    )
    return report


JSON_REPORT = grade_report("json")
summary = subprocess.run(
    [str(KALANOS), "inspect", str(JSON_REPORT)],
    capture_output=True,
    text=True,
    check=True,
)
# The full finding records (JSON lines) are kept in summary.txt and audit.json.
shown = summary.stdout.splitlines()
first_record = next((i for i, x in enumerate(shown) if x.startswith("{")), len(shown))
print("\n".join(shown[:first_record]))
(OUTPUT / "summary.txt").write_text(summary.stdout)

## 4. View the HTML report
This invokes a second assessment of the same pinned local files to produce HTML. It is not a JSON-to-HTML conversion. If the run includes sampled checks, inspect each report's run metadata before comparing them.

Read the score together with episode decisions, measured findings, missing evidence and scope. Undefined readiness is not zero and is not a pass. The fixture name describes injected intent; it is not proof that every injected event was detected.


In [ ]:
HTML_REPORT = grade_report("html")
# Notebook rendering may limit scripts; the downloaded HTML is the fallback.
display(HTML(HTML_REPORT.read_text()))

## 5. Download results
The ZIP includes reports, CLI summaries/logs, resolved dependency versions and the pinned dataset revision. It does not upload to Kalanos. Keep the provenance with reports when sharing results.

In [ ]:
BUNDLE = WORK / "kalanos-audit-results.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.iterdir():
        if path.is_file():
            archive.write(path, path.name)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: download", BUNDLE)
    display(FileLink(str(BUNDLE)))
else:
    files.download(str(BUNDLE))

## What to investigate next
- Which episodes and channels were flagged, and what measured evidence supports each finding?
- Did the available checks cover the synthetic fault you expected? Missing coverage is not a pass.
- Which findings are report-only versus review or blocking decisions?
- If you try the clean fixture, use `kalanosai/synthetic-clean-signals`, rerun from the start, and save its own revision. Do not compare scores across different scopes without qualification.

[Kalanos](https://kalanos.ai) · [Grader source and methodology](https://github.com/KalanosAI/kalanos) · [Public reports](https://hub.kalanos.ai) · [Synthetic fixture generator](https://hub.kalanos.ai/synthetic-data) (sign-in required)
